In [3]:
# run refresh_trade_views to init arbitrage tables
from db.auto_repo_sqlite import snapshot_many, TableSpec, upsert_many
from domain.market_rows import MarketGoodRow, MarketTransactionRow
import sqlite3
from services.arbitrage_repo import top_arbitrage

# get highest arbitrage data

ta = top_arbitrage()

#print(ta.iloc[0,0])
arbi_trade_symbol = ta.iloc[0,0]
arbi_buy_wp = ta.iloc[0,1]
arbi_buy_price = ta.iloc[0,2]
arbi_sell_wp = ta.iloc[0,3]
arbi_sell_price = ta.iloc[0,4]

print("This is the trade symbol: ", arbi_trade_symbol)
print("This is the buy waypoint: ", arbi_buy_wp)
print("This is the buy price: ", arbi_buy_price)
print("This is the sell waypoint: ", arbi_sell_wp)
print("This is the sell price: ", arbi_sell_price)
print(ta)

This is the trade symbol:  MEDICINE
This is the buy waypoint:  X1-ZP92-D40
This is the buy price:  2478
This is the sell waypoint:  X1-ZP92-J57
This is the sell price:  4899
        trade_symbol buy_waypoint  buy_price sell_waypoint  sell_price  delta  \
0           MEDICINE  X1-ZP92-D40       2478   X1-ZP92-J57        4899   2421   
1     ASSAULT_RIFLES  X1-ZP92-E42       2369   X1-ZP92-J57        4256   1887   
2    MICROPROCESSORS   X1-ZP92-A3       2336   X1-ZP92-D41        3999   1663   
3          EQUIPMENT  X1-ZP92-K86       2114   X1-ZP92-F48        3488   1374   
4            JEWELRY  X1-ZP92-H53       2067    X1-ZP92-A1        3291   1224   
5               GOLD   X1-ZP92-B7        239   X1-ZP92-H53         351    112   
6        URANITE_ORE   X1-ZP92-B7        322   X1-ZP92-F48         394     72   
7               FUEL  X1-ZP92-G49         53   X1-ZP92-C39          68     15   
8   SILICON_CRYSTALS   X1-ZP92-B7         37   X1-ZP92-F46          44      7   
9    LIQUID_NITR

In [4]:
from __future__ import annotations
import asyncio
from typing import Optional, List
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi
import math
import pandas as pd
from runtime_support import (
    setup_client_from_env,
    api_navigate_ship,
    api_get_ship_nav,
    api_purchase_cargo,
    build_fleet_object
    )
from core_helpers import (
    init_world_state
)

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)
    systems_api = SystemsApi(client)
from core_helpers import (
    init_world_state
)

with setup_client_from_env() as client:
    fleet_api = FleetApi(client)
    agents_api = AgentsApi(client)
    systems_api = SystemsApi(client)

#Build all in-memory objects once (fleet_activity_obj, waypoints_ref_obj, waypoint_traits_obj, HQ)
world_state = await init_world_state(fleet_api, agents_api, systems_api)
from adapters.ships_activity_adapter import merge_activity_with_nav
from adapters.ships_specs_adapter import adapt_ships_specs_from_ship
from core_helpers import adapt_ships_activity_from_ship, upsert_many
from typing import Any, Dict, Iterable, List, Optional, Tuple
from runtime_support import call_sdk, unwrap_data
from dataclasses import dataclass, field
from domain.ships_activity import ShipsActivity
from domain.ships_specs import ShipsSpecs
from domain.ship_market import ShipMarketRow
from openapi_client.api.fleet_api import FleetApi
from openapi_client.api.agents_api import AgentsApi
from openapi_client.api.systems_api import SystemsApi

In [5]:
@dataclass
class FleetState:
    """Local, easily-referenced state for your session."""
    # Activity & Specs keyed by ship symbol
    activities: Dict[str, ShipsActivity] = field(default_factory=dict)
    specs: Dict[str, ShipsSpecs] = field(default_factory=dict)

    # Shipyard listings cached by waypoint
    ship_market: Dict[str, List[ShipMarketRow]] = field(default_factory=dict)

    def ensure_activity(self, symbol: str) -> ShipsActivity:
        a = self.activities.get(symbol)
        if a is None:
            a = ShipsActivity(symbol=symbol)
            self.activities[symbol] = a
        return a

    def update_activity_from_nav(self, symbol: str, nav_dto: Any) -> ShipsActivity:
        current = self.ensure_activity(symbol)
        updated = merge_activity_with_nav(current, nav_dto)
        self.activities[symbol] = updated
        return updated

    def update_activity_from_refuel(self, symbol: str, resp_dto: Any) -> ShipsActivity:
        """Use refuel response to update fuel state in local activity."""
        current = self.ensure_activity(symbol)
        fuel_cur = getattr(getattr(resp_dto, "fuel", None), "current", None)
        fuel_cap = getattr(getattr(resp_dto, "fuel", None), "capacity", None)
        patched = current.model_copy(update={
            "fuel_current": fuel_cur if fuel_cur is not None else current.fuel_current,
            "fuel_capacity": fuel_cap if fuel_cap is not None else current.fuel_capacity,
        })
        self.activities[symbol] = patched
        return patched

In [6]:
async def local_fleet_state(fleet: FleetApi) -> FleetState:
    """Load ships once, build local state (activity + specs) and persist to DB."""
    resp = await call_sdk(fleet, "get_my_ships")
    ships: Iterable[Any] = unwrap_data(resp)
    ships = list(ships)
    if not ships:
        raise SystemExit("[FATAL] No ships returned; check token/agent.")

    activities = [adapt_ships_activity_from_ship(d) for d in ships]
    specs = [adapt_ships_specs_from_ship(d) for d in ships]
    state = FleetState(
        activities={a.symbol: a for a in activities if a.symbol},
        specs={s.symbol: s for s in specs if s.symbol},
    )
    return state

# --------- define ship roles -----------
state = await local_fleet_state(fleet_api)
print(state.specs)

# Extract symbol for a given role
def get_symbol_by_role(ships_dict, role):
    for ship in ships_dict.values():
        if ship.role == role:
            return ship.symbol
    return None

command_ship = get_symbol_by_role(state.specs, "COMMAND")
satellite = get_symbol_by_role(state.specs, "SATELLITE")
print(command_ship)

{'GLANK-1': ShipsSpecs(symbol='GLANK-1', role='COMMAND', frame_name='Frigate', frame_module_slots=8, frame_mounting_points=5, engine_name='Ion Drive II', speed=36, mounts=['MOUNT_SENSOR_ARRAY_II', 'MOUNT_GAS_SIPHON_II', 'MOUNT_MINING_LASER_II', 'MOUNT_SURVEYOR_II'], modules=['MODULE_CARGO_HOLD_II', 'MODULE_CREW_QUARTERS_I', 'MODULE_CREW_QUARTERS_I', 'MODULE_MINERAL_PROCESSOR_I', 'MODULE_GAS_PROCESSOR_I'], capacity=40), 'GLANK-2': ShipsSpecs(symbol='GLANK-2', role='SATELLITE', frame_name='Probe', frame_module_slots=0, frame_mounting_points=0, engine_name='Impulse Drive I', speed=9, mounts=[], modules=[], capacity=0)}
GLANK-1


In [7]:
await api_navigate_ship(fleet_api, command_ship, arbi_buy_wp)

starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Ship is already at the destination
Prep complete
[SKIP] Navigation aborted, already at destination


In [10]:
from runtime_support import api_dock_ship, api_sell_cargo, api_purchase_cargo
await api_dock_ship(fleet_api, command_ship)
await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp, arbi_trade_symbol, 20)

PurchaseCargo201ResponseData(cargo=ShipCargo(capacity=40, units=40, inventory=[ShipCargoItem(symbol=<TradeSymbol.MEDICINE: 'MEDICINE'>, name='Medicine', description='Medical products, including drugs, treatments, and medical equipment.', units=40)]), transaction=MarketTransaction(waypoint_symbol='X1-ZP92-D40', ship_symbol='GLANK-1', trade_symbol='MEDICINE', type='PURCHASE', units=20, price_per_unit=2945, total_price=58900, timestamp=datetime.datetime(2025, 9, 16, 19, 24, 42, 840000, tzinfo=TzInfo(UTC))), agent=Agent(account_id='cmeb98xyw0028tm167bfnq149', symbol='GLANK', headquarters='X1-ZP92-A1', credits=148155, starting_faction='AEGIS', ship_count=2))

In [15]:
from typing import Dict, List
from refuel_routing import Node
from domain.waypoint_trait import WaypointTraitRow

def build_nodes_from_traits_dict(
    traits_dict: Dict[str, List[WaypointTraitRow]],
    fuel_price_lookup: Dict[str, float] = None,
) -> List[Node]:
    """
    Convert a dict of {waypoint_symbol: [WaypointTraitRow, ...]} into Node objects.

    Args:
        traits_dict: mapping from waypoint_symbol -> list of WaypointTraitRow.
        fuel_price_lookup: optional {waypoint_symbol: price} for refining
                           Node.price (default None).

    Returns:
        List[Node]
    """
    nodes: List[Node] = []

    for symbol, trait_rows in traits_dict.items():
        if not trait_rows:
            continue  # skip empty

        # all rows for this waypoint share x,y,type
        first = trait_rows[0]
        x, y = float(first.x), float(first.y)

        # check if any trait is a MARKETPLACE
        has_marketplace = any(tr.trait_symbol == "MARKETPLACE" for tr in trait_rows)

        # build Node
        node = Node(
            symbol=symbol,
            x=x,
            y=y,
            has_fuel=has_marketplace,
            price=(fuel_price_lookup.get(symbol) if fuel_price_lookup else None),
        )
        nodes.append(node)

    return nodes

world_state = await init_world_state(fleet_api, agents_api, systems_api)

wps = world_state.traits.by_wp

nodes = build_nodes_from_traits_dict(wps)
from refuel_routing import (
    Node,
    build_candidates,
    build_reachability_graph,
    astar_by_time,
    compute_refuel_plan,
    corridor_polygon,
    plan_route_and_refuel,
)


plan_to_buy = plan_route_and_refuel(nodes, arbi_sell_wp, arbi_buy_wp, 400, 1, 1000, 400, 50, 0, 0, 0)
plan_to_sell = plan_route_and_refuel(nodes, arbi_buy_wp, arbi_sell_wp, 400, 1, 1000, 400, 50, 0, 0, 0)

In [16]:
route_buy_point = plan_to_buy["path"]
route_sell_point = plan_to_sell["path"]

In [ ]:
for rt in route_buy_point:
    await api_navigate_ship(fleet_api, command_ship, rt)

await api_dock_ship(fleet_api, command_ship)
await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp, arbi_trade_symbol, 20)
await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp, arbi_trade_symbol, 20)

for rt in route_sell_point:
    await api_navigate_ship(fleet_api, command_ship, rt)

await api_dock_ship(fleet_api, command_ship)
await api_sell_cargo(fleet_api, command_ship, arbi_sell_wp, arbi_trade_symbol, 20)
await api_sell_cargo(fleet_api, command_ship, arbi_sell_wp, arbi_trade_symbol, 20)

for rt in route_buy_point:
    await api_navigate_ship(fleet_api, command_ship, rt)

await api_dock_ship(fleet_api, command_ship)
await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp, arbi_trade_symbol, 20)
await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp, arbi_trade_symbol, 20)

for rt in route_sell_point:
    await api_navigate_ship(fleet_api, command_ship, rt)

await api_dock_ship(fleet_api, command_ship)
await api_sell_cargo(fleet_api, command_ship, arbi_sell_wp, arbi_trade_symbol, 20)
await api_sell_cargo(fleet_api, command_ship, arbi_sell_wp, arbi_trade_symbol, 20)

for rt in route_buy_point:
    await api_navigate_ship(fleet_api, command_ship, rt)

await api_dock_ship(fleet_api, command_ship)
await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp, arbi_trade_symbol, 20)
await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp, arbi_trade_symbol, 20)

for rt in route_sell_point:
    await api_navigate_ship(fleet_api, command_ship, rt)

await api_dock_ship(fleet_api, command_ship)
await api_sell_cargo(fleet_api, command_ship, arbi_sell_wp, arbi_trade_symbol, 20)
await api_sell_cargo(fleet_api, command_ship, arbi_sell_wp, arbi_trade_symbol, 20)

for rt in route_buy_point:
    await api_navigate_ship(fleet_api, command_ship, rt)

await api_dock_ship(fleet_api, command_ship)
await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp, arbi_trade_symbol, 20)
await api_purchase_cargo(fleet_api, command_ship, arbi_buy_wp, arbi_trade_symbol, 20)

for rt in route_sell_point:
    await api_navigate_ship(fleet_api, command_ship, rt)

await api_dock_ship(fleet_api, command_ship)
await api_sell_cargo(fleet_api, command_ship, arbi_sell_wp, arbi_trade_symbol, 20)
await api_sell_cargo(fleet_api, command_ship, arbi_sell_wp, arbi_trade_symbol, 20)

starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Ship is already at the destination
Prep complete
[SKIP] Navigation aborted, already at destination
starting navigation...
[BOOT] Adapted 2 ships into fleet_object
Not at destination, continuing
Refuelling now...
Not in orbit, going into orbit now...
Prep complete
GLANK-1  has taken off and is in transit
[BOOT] Adapted 2 ships into fleet_object
Seconds until arrival: 98.640255
